In [9]:
import pandas as pd
import numpy as np
import csv


In [10]:
def hampel_filter(data, window_size=3, n_sigmas=3.0):
    s = pd.Series(data)
    rolling_median = s.rolling(window=window_size, min_periods=1, center=True).median()
    rolling_mad = (s - rolling_median).abs().rolling(window=window_size, min_periods=1, center=True).median()
    threshold = n_sigmas * 1.4826 * rolling_mad
    difference = (s - rolling_median).abs()
    outliers = difference > threshold
    s_filtered = s.copy()
    s_filtered[outliers] = rolling_median[outliers]
    return s_filtered.tolist()

In [11]:
lat = []
lon = []
with open("2026-09-12_route00_gnss.csv", "r") as gnss_csv:
    gnss_list = csv.reader(gnss_csv)
    gnss_list = np.array(list(gnss_list))
    lat = gnss_list[1:, 0]
    lon = gnss_list[1:, 1]

lat = np.float32(lat[500:1000])
lon = np.float32(lon[500:1000])

In [14]:
def hampel_filter(lat, lon, window_size=3, n_sigmas=3.0):
    lat = np.asarray(lat)
    lon = np.asarray(lon)
    n = len(lat)

    k = 1.4826
    outlier_mask = np.zeros(n, dtype=bool)

    for i in range(n):
        start = max(0, i - window_size)
        end = min(n, i + window_size + 1)

        win_lat = lat[start:end]
        med_lat = np.median(win_lat)
        mad_lat = np.median(np.abs(win_lat - med_lat))
        thresh_lat = k * mad_lat * n_sigmas

        is_lat_outlier = (thresh_lat > 0) and (np.abs(lat[i] - med_lat) > thresh_lat)

        win_lon = lon[start:end]
        med_lon = np.median(win_lon)
        mad_lon = np.median(np.abs(win_lon - med_lon))
        thresh_lon = k * mad_lon * n_sigmas

        is_lon_outlier = (thresh_lon > 0) and (np.abs(lon[i] - med_lon) > thresh_lon)

        if is_lat_outlier or is_lon_outlier:
            outlier_mask[i] = True

    clean_lat = lat[~outlier_mask]
    clean_lon = lon[~outlier_mask]
    outlier_indices = np.where(outlier_mask)[0].tolist()

    return clean_lat, clean_lon, outlier_indices

filtered_lat, filtered_lon, dropped_indices = hampel_filter(
    lat, lon, window_size=3, n_sigmas=3.0
)

In [15]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=filtered_lon, y=filtered_lat,
    mode='lines+markers',
    name='points',
    line=dict(color='blue', width=1),
    marker=dict(size=8, color='red'),
    text=[f"Index: {idx}<br>Lat: {lat:.6f}<br>Lon: {lon:.6f}"
            for idx, (lat, lon) in enumerate(zip(filtered_lat, filtered_lon))],
    hoverinfo='text'
))
fig.update_layout(
    title="Points from filtered latlon",
    xaxis_title='Longitude (deg)',
    yaxis_title='Latitude (deg)',
    hovermode='closest',
    width=1000, height=1000,
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)

output_file = "check_latlon_filtered.html"
fig.write_html(output_file)


fig = go.Figure()
fig.add_trace(go.Scatter(
    x=lon, y=lat,
    mode='lines+markers',
    name='points',
    line=dict(color='blue', width=1),
    marker=dict(size=8, color='red'),
    text=[f"Index: {idx}<br>Lat: {l_val:.6f}<br>Lon: {r_val:.6f}"
            for idx, (l_val, r_val) in enumerate(zip(lat, lon))],
    hoverinfo='text'
))
fig.update_layout(
    title="Points from dirty latlon",
    xaxis_title='Longitude (deg)',
    yaxis_title='Latitude (deg)',
    hovermode='closest',
    width=1000, height=1000,
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)

output_file = "check_latlon_dirty.html"
fig.write_html(output_file)

In [16]:
print("len clean lat: ",len(filtered_lat))
print("len dirty lat: ",len(lat))


len clean lat:  399
len dirty lat:  500


In [17]:
import time
import numpy as np

def latlon_stream_generator(lats, lons, delay_seconds=0.1):
    for lat, lon in zip(lats, lons):
        time.sleep(delay_seconds)
        yield lat, lon

def run_streaming_hampel_simulation(lats, lons, window_size=3, n_sigmas=3.0, delay_seconds=0.1):
    k = 1.4826
    buffer_lat = []
    buffer_lon = []

    clean_lat = []
    clean_lon = []
    outliers = []

    stream = latlon_stream_generator(lats, lons, delay_seconds)

    for idx, (lat, lon) in enumerate(stream):
        buffer_lat.append(lat)
        buffer_lon.append(lon)

        max_len = 2 * window_size + 1
        if len(buffer_lat) > max_len:
            buffer_lat.pop(0)
            buffer_lon.pop(0)

        if len(buffer_lat) < max_len:
            clean_lat.append(lat)
            clean_lon.append(lon)
            print(f"[STREAM] Index {idx:02d} | Lat: {lat:.6f}, Lon: {lon:.6f} | Status: BUFFERING")
            continue

        mid_idx = window_size
        target_lat = buffer_lat[mid_idx]
        target_lon = buffer_lon[mid_idx]
        global_idx = idx - window_size

        win_lat = np.array(buffer_lat)
        win_lon = np.array(buffer_lon)

        med_lat = np.median(win_lat)
        mad_lat = np.median(np.abs(win_lat - med_lat))
        thresh_lat = k * mad_lat * n_sigmas

        med_lon = np.median(win_lon)
        mad_lon = np.median(np.abs(win_lon - med_lon))
        thresh_lon = k * mad_lon * n_sigmas

        is_lat_outlier = (thresh_lat > 0) and (np.abs(target_lat - med_lat) > thresh_lat)
        is_lon_outlier = (thresh_lon > 0) and (np.abs(target_lon - med_lon) > thresh_lon)

        if is_lat_outlier or is_lon_outlier:
            outliers.append(global_idx)
            print(f"[STREAM] Index {global_idx:02d} | Lat: {target_lat:.6f}, Lon: {target_lon:.6f} | Status: OUTLIER REJECTED")
        else:
            clean_lat.append(target_lat)
            clean_lon.append(target_lon)
            print(f"[STREAM] Index {global_idx:02d} | Lat: {target_lat:.6f}, Lon: {target_lon:.6f} | Status: CLEAN")

    return clean_lat, clean_lon, outliers

# --- Execution using your existing lat and lon lists ---
cleaned_lat, cleaned_lon, detected_outliers = run_streaming_hampel_simulation(lat, lon, window_size=3, n_sigmas=3.0, delay_seconds=0.05)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=cleaned_lon, y=cleaned_lat,
    mode='lines+markers',
    name='points',
    line=dict(color='blue', width=1),
    marker=dict(size=8, color='red'),
    text=[f"Index: {idx}<br>Lat: {lat:.6f}<br>Lon: {lon:.6f}"
            for idx, (lat, lon) in enumerate(zip(cleaned_lat, cleaned_lon))],
    hoverinfo='text'
))
fig.update_layout(
    title="Points from cleaned latlon",
    xaxis_title='Longitude (deg)',
    yaxis_title='Latitude (deg)',
    hovermode='closest',
    width=1000, height=1000,
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)

output_file = "check_latlon_filtered.html"
fig.write_html(output_file)

[STREAM] Index 00 | Lat: -7.768392, Lon: 110.384277 | Status: BUFFERING
[STREAM] Index 01 | Lat: -7.768370, Lon: 110.384117 | Status: BUFFERING
[STREAM] Index 02 | Lat: -7.768393, Lon: 110.384285 | Status: BUFFERING
[STREAM] Index 03 | Lat: -7.768393, Lon: 110.384293 | Status: BUFFERING
[STREAM] Index 04 | Lat: -7.768393, Lon: 110.384308 | Status: BUFFERING
[STREAM] Index 05 | Lat: -7.768393, Lon: 110.384331 | Status: BUFFERING
[STREAM] Index 03 | Lat: -7.768393, Lon: 110.384293 | Status: CLEAN
[STREAM] Index 04 | Lat: -7.768393, Lon: 110.384308 | Status: CLEAN
[STREAM] Index 05 | Lat: -7.768393, Lon: 110.384331 | Status: CLEAN
[STREAM] Index 06 | Lat: -7.768372, Lon: 110.384132 | Status: OUTLIER REJECTED
[STREAM] Index 07 | Lat: -7.768393, Lon: 110.384338 | Status: CLEAN
[STREAM] Index 08 | Lat: -7.768394, Lon: 110.384354 | Status: CLEAN
[STREAM] Index 09 | Lat: -7.768394, Lon: 110.384361 | Status: CLEAN
[STREAM] Index 10 | Lat: -7.768393, Lon: 110.384384 | Status: CLEAN
[STREAM] Inde